# Pokrytí mamografickým screeningem: kde je účinek, který celek skrývá

Revize i rozšíření se dívaly hlavně na celou populaci žen 45+. Řada ukazatelů tam s pokrytím
nesouvisí: vzdálenost k pracovišti, kapacita v dosahu, prevence u praktika, velikost obcí,
socioekonomika. Nulový vztah v celku ale nevylučuje silný vztah v části populace, který se
v průměru ztratí, nebo vztahy opačného směru v různých skupinách, které se vyruší.

Hledání podskupin je ale nejjistější cesta k falešným nálezům: při stovkách testů vyjdou
„průkazné“ podskupiny i z čistého šumu. Proto tu platí čtyři pojistky:

1. **Jeden velký sken s korekcí na všechny testy najednou** (Benjamini–Hochberg přes všechny
   ukazatele × všechny podskupiny), ne výběr toho, co vyšlo.
2. **Replikace v jiném období:** podskupinu najdeme na pokrytí 2014–2018 a ověříme na 2020–2024.
   Ukazatele okresů jsou v obou obdobích stejné a pokrytí je dost stálé, takže replikace hlídá
   hlavně šum jednotlivých let, ne celou nejistotu.
3. **Test interakce:** podskupina se počítá, jen když se její vztah průkazně liší od zbytku,
   ne jen když v ní vyjde p < 0,05.
4. **Předem formulovaná predikce:** co sken najde, ověříme na jiném typu dat (změna v čase,
   otevření pracoviště).

Podskupiny: **věková pásma** (pokrytí známe po pětiletých pásmech) a **typy okresů** (městské a
venkovské, s vlastním pracovištěm a bez něj, vyšší a nižší nezaměstnanost, Čechy a Morava,
blízko a daleko od pracoviště). Analýza zůstává **ekologická**.

In [1]:
import sys
import warnings
from pathlib import Path

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "nacteni.py").exists())
sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests

from utils import analyza as an
from utils import nacteni as nac
from utils.vzhled import AXIS, DIV, GRID, INK2, MUTED, S1, S2, SURFACE, cz, layout

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"
OUT = BASE / "outputs" / "podskupiny"
OUT.mkdir(parents=True, exist_ok=True)

V = nac.vse()
D, X, CENTROID, centra, OCK = (V[k] for k in ("D", "X", "CENTROID", "centra", "OCK"))
LAUS = sorted(CENTROID)
X = X.set_index("lau").reindex(LAUS).join(OCK.set_index("lau"))
X["prev_index"] = an.prev_index(X)
clon = np.array([CENTROID[l][0] for l in LAUS]); clat = np.array([CENTROID[l][1] for l in LAUS])
DIST = nac.haversine(clon[:, None], clat[:, None], centra.lon.values[None, :], centra.lat.values[None, :])
popt = D[D.band.astype(str).isin(nac.BANDS[:6]) & D.rok.isin(nac.ROKY_V)].groupby("lau").zeny.mean().reindex(LAUS)
X["sfca_30"] = an.sfca(DIST, popt, 30)
gyn = nac.gyn_okresy().set_index("lau").reindex(LAUS)
X["gyn_ord_10k"] = 1e4 * gyn.gyn_ordinace / X.zeny_celkem

POPIS = {**nac.POPIS, "sfca_30": ("pracoviště v dosahu 30 km", ""), "gyn_ord_10k": ("ordinace gynekologů na 10 000 žen", ""),
         "chripka65_pred": ("chřipka 65+ před covidem", "%"), "chripka65_po": ("chřipka 65+ po covidu", "%"),
         "covid_1_zeny45": ("covid 1. dávka, ženy 45+", "%"), "covid_3_zeny45": ("covid 3. dávka, ženy 45+", "%"),
         "prev_pasmo": ("prevence u praktika ve stejném pásmu", "%")}
EXPO = [c for c in nac.POPIS if c in X.columns and c not in an.DUSLEDKY] + \
       ["sfca_30", "gyn_ord_10k", "chripka65_pred", "chripka65_po", "covid_1_zeny45", "covid_3_zeny45"]
OBDOBI = {"objev": range(2014, 2019), "ověření": range(2020, 2025)}
print(f"{len(EXPO)} ukazatelů okresů + prevence u praktika po pásmech · 9 věkových pásem · 10 typů okresů")

47 ukazatelů okresů + prevence u praktika po pásmech · 9 věkových pásem · 10 typů okresů


## 1 · Sken: ukazatel × podskupina

Pro každé věkové pásmo korelujeme ukazatel okresu s pokrytím mamografií v tom pásmu (dvouleté
okno, počty sečtené za období). Pro typy okresů korelujeme s věkově standardizovaným pokrytím
uvnitř dané poloviny okresů. Prevenci u praktika bereme ve stejném věkovém pásmu jako mamografii.

In [2]:
def pasma_obdobi(roky):
    P = an.pasma(D, roky)
    return (P.pivot(index="lau", columns="band", values="m").reindex(LAUS),
            P.pivot(index="lau", columns="band", values="p").reindex(LAUS),
            an.standardizuj(P, "m", an.standard(P)).reindex(LAUS))

PO = {k: pasma_obdobi(r) for k, r in OBDOBI.items()}
MOD = {"městské okresy": X.obce_4_pct > X.obce_4_pct.median(),
       "venkovské okresy": X.obce_4_pct <= X.obce_4_pct.median(),
       "s vlastním pracovištěm": X.ma_mamocentrum == 1, "bez vlastního pracoviště": X.ma_mamocentrum == 0,
       "vyšší nezaměstnanost": X.nezam_zeny_pct > X.nezam_zeny_pct.median(),
       "nižší nezaměstnanost": X.nezam_zeny_pct <= X.nezam_zeny_pct.median(),
       "Čechy": ~X.index.str[:4].isin(["CZ06", "CZ07", "CZ08"]),
       "Morava a Slezsko": X.index.str[:4].isin(["CZ06", "CZ07", "CZ08"]),
       "daleko od pracoviště (> 15 km)": X.nej_km > 15, "blízko pracoviště (≤ 15 km)": X.nej_km <= 15}

def kor(x, y):
    ok = x.notna() & y.notna()
    if ok.sum() < 15 or x[ok].std() == 0:
        return np.nan, np.nan, int(ok.sum())
    r, p = stats.pearsonr(x[ok], y[ok])
    return r, p, int(ok.sum())

rows = []
for b in nac.BANDS:
    for c in EXPO + ["prev_pasmo"]:
        rec = {"typ": "věkové pásmo", "podskupina": b, "ukazatel": POPIS[c][0], "kod": c}
        for k, (mb, pb, _) in PO.items():
            x = pb[b] if c == "prev_pasmo" else X[c]
            rec[f"r {k}"], rec[f"p {k}"], rec["n"] = kor(x, mb[b])
        rows.append(rec)
for nm, mask in MOD.items():
    for c in EXPO:
        rec = {"typ": "typ okresu", "podskupina": nm, "ukazatel": POPIS[c][0], "kod": c}
        for k, (_, _, ms) in PO.items():
            rec[f"r {k}"], rec[f"p {k}"], rec["n"] = kor(X[c][mask], ms[mask])
        rows.append(rec)
SKEN = pd.DataFrame(rows).dropna(subset=["p objev"])
SKEN["q objev"] = multipletests(SKEN["p objev"], method="fdr_bh")[1]
SKEN["replikováno"] = (SKEN["q objev"] < 0.05) & (np.sign(SKEN["r objev"]) == np.sign(SKEN["r ověření"])) & (SKEN["p ověření"] < 0.05)

# celek: stejný ukazatel × standardizované pokrytí všech okresů a všech věků (období objevu)
celek = {c: kor(X[c], PO["objev"][2]) for c in EXPO}
SKEN["r celek"] = [celek[c][0] if c in celek else np.nan for c in SKEN.kod]
SKEN["p celek"] = [celek[c][1] if c in celek else np.nan for c in SKEN.kod]
SKEN.to_csv(OUT / "sken_podskupin.csv", index=False)
print(f"testů: {len(SKEN)} · q < 0,05 v období objevu: {int((SKEN['q objev'] < 0.05).sum())} · "
      f"z toho replikováno v 2020–2024: {int(SKEN['replikováno'].sum())}")
SKRYTE = SKEN[SKEN["replikováno"] & ((SKEN["p celek"] > 0.05) | SKEN["kod"].eq("prev_pasmo"))]
print(f"replikované podskupiny u ukazatelů, které v celku průkazné NEJSOU: {len(SKRYTE)}")
SKRYTE.sort_values("q objev")[["typ", "podskupina", "ukazatel", "n", "r celek", "r objev", "q objev", "r ověření", "p ověření"]].round(3)

testů: 897 · q < 0,05 v období objevu: 69 · z toho replikováno v 2020–2024: 67
replikované podskupiny u ukazatelů, které v celku průkazné NEJSOU: 17


,typ,podskupina,ukazatel,n,r celek,r objev,q objev,r ověření,p ověření
407,věkové pásmo,85+,vzdálenost k nejbližšímu pracovišti,77,-0.176,-0.533,0.000,-0.490,0.000
359,věkové pásmo,80–84,vzdálenost k nejbližšímu pracovišti,77,-0.176,-0.525,0.000,-0.472,0.000
405,věkové pásmo,85+,obyvatelé obcí 50 000+ obyv.,77,-0.056,0.442,0.003,0.513,0.000
357,věkové pásmo,80–84,obyvatelé obcí 50 000+ obyv.,77,-0.056,0.430,0.004,0.458,0.000
311,věkové pásmo,75–79,vzdálenost k nejbližšímu pracovišti,77,-0.176,-0.414,0.007,-0.334,0.003
425,věkové pásmo,85+,pracoviště v dosahu 30 km,77,0.220,0.401,0.010,0.241,0.035
354,věkové pásmo,80–84,obyvatelé obcí 500–1 999 obyv.,77,0.054,-0.392,0.013,-0.372,0.001
377,věkové pásmo,80–84,pracoviště v dosahu 30 km,77,0.220,0.389,0.014,0.242,0.034
797,typ okresu,Morava a Slezsko,volby 2025: STAN,27,-0.080,0.621,0.014,0.570,0.002
402,věkové pásmo,85+,obyvatelé obcí 500–1 999 obyv.,77,0.054,-0.368,0.022,-0.432,0.000


Většina replikovaných nálezů jsou ukazatele, které s pokrytím souvisejí už v celku (gynekologická
prevence, test na okultní krvácení); ty tu nejsou zajímavé. Tabulka výše ukazuje jen ty, které
v celku průkazné nejsou. Na první pohled se v ní opakuje jedno téma: **dostupnost a věk**.
Následující mapa korelací po věkových pásmech to ukazuje pro vybrané ukazatele.

In [3]:
VYBER = ["vzdalenost_km", "ma_mamocentrum", "dojizdka_mamo_pct", "sfca_30", "obce_4_pct", "obce_0_pct",
         "hustota_obyv_km2", "cizinky_pct", "mimo_pracovni_silu_zeny_pct", "vs_zeny_pct", "gyn_prevence_pct", "toks_pct"]
Hm = SKEN[(SKEN.typ == "věkové pásmo") & SKEN.kod.isin(VYBER)].pivot(index="kod", columns="podskupina", values="r ověření").reindex(index=VYBER, columns=nac.BANDS)
Hq = SKEN[(SKEN.typ == "věkové pásmo") & SKEN.kod.isin(VYBER)].pivot(index="kod", columns="podskupina", values="replikováno").reindex(index=VYBER, columns=nac.BANDS)
txt = [[f"{cz(Hm.iat[i, j], 2)}{'*' if Hq.iat[i, j] else ''}" for j in range(Hm.shape[1])] for i in range(Hm.shape[0])]
hf = go.Figure(go.Heatmap(z=Hm.values, x=nac.BANDS, y=[POPIS[c][0] for c in VYBER], text=txt, texttemplate="%{text}",
                          colorscale=DIV, zmin=-0.6, zmax=0.6, xgap=2, ygap=2,
                          colorbar=dict(title=dict(text="r", side="top"), thickness=12, len=0.6, outlinewidth=0)))
hf.update_layout(**layout("Dostupnost rozhoduje u nejstarších žen",
                          "korelace ukazatele okresu s pokrytím daného věkového pásma, 2020–2024 · "
                          "* nalezeno v 2014–2018 po korekci BH a replikováno"))
hf.update_layout(height=520, margin=dict(l=280, r=30, t=90, b=50),
                 yaxis=dict(autorange="reversed", showgrid=False, tickfont=dict(color=INK2)),
                 xaxis=dict(side="top", showgrid=False))
hf.write_html(OUT / "pasma_heatmap.html", include_plotlyjs="cdn")
hf

## 2 · Test interakce: liší se nejstarší ženy průkazně?

Že vztah v pásmu 80–84 vyjde průkazně a v pásmu 50–54 ne, ještě neznamená, že se pásma liší.
Proto pro **každý** ukazatel odhadneme jeden model nad všemi okresy a pásmy (pokrytí standardizované
uvnitř pásma) s interakcí „ukazatel × věk 75+“ a chybami shlukovanými po okresech. Korekce BH
běží přes všechny ukazatele.

In [4]:
mb_v = PO["ověření"][0]
L = mb_v.stack().rename("m").reset_index()
L["z"] = L.groupby("band").m.transform(lambda s: (s - s.mean()) / s.std())
L["stari"] = L.band.isin(["75–79", "80–84", "85+"]).astype(int)
rows = []
for c in EXPO:
    L["x"] = L.lau.map((X[c] - X[c].mean()) / X[c].std())
    q = L.dropna(subset=["x", "z"])
    m = smf.ols("z ~ x * stari", q).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(q.lau)[0]})
    rows.append({"ukazatel": POPIS[c][0], "kod": c, "sklon 45–74": m.params["x"], "p 45–74": m.pvalues["x"],
                 "sklon 75+": m.params["x"] + m.params["x:stari"], "interakce": m.params["x:stari"],
                 "p interakce": m.pvalues["x:stari"]})
INT = pd.DataFrame(rows)
INT["q interakce"] = multipletests(INT["p interakce"], method="fdr_bh")[1]
INT.to_csv(OUT / "interakce_vek75.csv", index=False)
print(f"ukazatelů s průkaznou interakcí s věkem 75+ (q < 0,05): {int((INT['q interakce'] < 0.05).sum())} z {len(INT)}")
INT[INT["q interakce"] < 0.05].sort_values("p interakce").drop(columns="kod").round(3)

ukazatelů s průkaznou interakcí s věkem 75+ (q < 0,05): 20 z 47


,ukazatel,sklon 45–74,p 45–74,sklon 75+,interakce,p interakce,q interakce
8,ženy s cizím občanstvím,-0.316,0.001,0.195,0.511,0.000,0.000
3,ženy mimo pracovní sílu,0.196,0.062,-0.342,-0.538,0.000,0.000
21,obyvatelé obcí 50 000+ obyv.,-0.028,0.759,0.414,0.443,0.000,0.000
17,obyvatelé obcí do 499 obyv.,0.266,0.022,-0.252,-0.518,0.000,0.000
11,hustota zalidnění,-0.166,0.000,0.246,0.412,0.000,0.000
18,obyvatelé obcí 500–1 999 obyv.,0.132,0.123,-0.326,-0.458,0.000,0.000
23,vzdálenost k nejbližšímu pracovišti,-0.042,0.663,-0.432,-0.390,0.000,0.000
37,volby 2025: Piráti,-0.039,0.711,0.305,0.343,0.000,0.000
5,příspěvek na bydlení na obyvatele,-0.241,0.005,0.194,0.435,0.000,0.000
4,dávky SSP na obyvatele,-0.160,0.054,0.199,0.359,0.000,0.000


Sklony jsou v jednotkách směrodatné odchylky pokrytí v pásmu na jednu směrodatnou odchylku
ukazatele (2020–2024).

## 3 · Změny v čase po věkových pásmech

Průřez může odrážet trvalé rozdíly mezi okresy. Silnější test je, jestli v podskupině působí i
**změna** uvnitř okresu. Pro každé pásmo zvlášť (pevné efekty okresu a roku, chyby po okresech):

* prevence u praktika ve stejném pásmu → mamografie v tom pásmu, 2011–2024 (v celku nulový vztah),
* objednací lhůta pracoviště ve vlastním okrese → mamografie (snímky 2013–2018; v celku nulový vztah).

Korekce BH přes všech 18 odhadů.

In [5]:
Bp = D[D.band != nac.TOTAL].copy()
Bp["band"] = Bp.band.astype(str)
Bp["m1"] = 100 * Bp.pocet_vysetreni.fillna(0) / Bp.zeny
Bp["p1"] = 100 * Bp.pocet_vysetrenych / Bp.populace
cek = nac.cekaci_doby().groupby(["lau", "rok"]).cekani_dni.mean().rename("cek").reset_index()

def fe(formule, q, var, nasob=1):
    m = smf.ols(formule + " + C(lau) + C(rok)", q).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(q.lau)[0]})
    return nasob * m.params[var], nasob * m.bse[var], m.pvalues[var], int(m.nobs)

rows = []
for b in nac.BANDS:
    q = Bp[(Bp.band == b) & Bp.rok.between(2011, 2024)].dropna(subset=["m1", "p1"])
    rows.append(("prevence u praktika, +1 p. b.", b, *fe("m1 ~ p1", q, "p1")))
    q2 = Bp[Bp.band == b].merge(cek, on=["lau", "rok"]).dropna(subset=["m1", "cek"])
    rows.append(("objednací lhůta, +10 dní", b, *fe("m1 ~ cek", q2, "cek", 10)))
DYN = pd.DataFrame(rows, columns=["expozice", "pásmo", "efekt p. b.", "SE", "p", "n"])
DYN["q"] = multipletests(DYN.p, method="fdr_bh")[1]
DYN.to_csv(OUT / "zmeny_v_case_pasma.csv", index=False)
print(f"nominálně p < 0,05: {int((DYN.p < 0.05).sum())} z {len(DYN)} · po korekci BH (q < 0,05): {int((DYN.q < 0.05).sum())}")
DYN.round(3)

nominálně p < 0,05: 1 z 18 · po korekci BH (q < 0,05): 0


,expozice,pásmo,efekt p. b.,SE,p,n,q
0,"prevence u praktika, +1 p. b.",45–49,0.039,0.036,0.277,1078,0.946
1,"objednací lhůta, +10 dní",45–49,0.078,0.110,0.476,251,0.946
2,"prevence u praktika, +1 p. b.",50–54,0.014,0.046,0.753,1078,0.946
3,"objednací lhůta, +10 dní",50–54,0.016,0.144,0.912,251,0.946
4,"prevence u praktika, +1 p. b.",55–59,-0.033,0.040,0.409,1078,0.946
5,"objednací lhůta, +10 dní",55–59,0.037,0.122,0.762,251,0.946
6,"prevence u praktika, +1 p. b.",60–64,-0.017,0.036,0.645,1078,0.946
7,"objednací lhůta, +10 dní",60–64,-0.011,0.136,0.934,251,0.946
8,"prevence u praktika, +1 p. b.",65–69,-0.004,0.038,0.924,1078,0.946
9,"objednací lhůta, +10 dní",65–69,0.032,0.129,0.803,251,0.946


## 4 · Předem daná predikce: pomohlo otevření pracoviště hlavně starším ženám?

Pokud dostupnost rozhoduje hlavně u žen 75+, mělo by první pracoviště v Teplicích (2016, viz
`mamograf_rozsireni`, kap. 3) zvednout pokrytí víc u starších než u mladších žen. Predikci jsme
formulovali z průřezu (kap. 1–2) dřív, než jsme se na Teplice po pásmech podívali.

Pro každé pásmo: rozdíl Teplic proti průměru ostatních okresů Ústeckého kraje před 2016 a od 2016
(rozdíl v rozdílech) a placebo test se stejným výpočtem pro všechny ostatní okresy.

In [6]:
wide_b = Bp[Bp.rok.between(2010, 2024)].pivot_table(index=["band", "lau"], columns="rok", values="m1")

def did_pasmo(b, l, rok0=2016):
    w = wide_b.loc[b]
    ctrl = [x for x in LAUS if x[:5] == l[:5] and x != l]
    gap = w.loc[l] - w.loc[ctrl].mean()
    return gap[gap.index >= rok0].mean() - gap[gap.index < rok0].mean()

rows = []
for b in nac.BANDS:
    e = did_pasmo(b, "CZ0426")
    plac = np.array([did_pasmo(b, l) for l in LAUS if l not in ("CZ0426", "CZ0100")])
    rows.append({"pásmo": b, "efekt Teplice p. b.": e, "placebo p (jednostranně)": (1 + (plac >= e).sum()) / (len(plac) + 1),
                 "efekt relativně k úrovni pásma %": 100 * e / wide_b.loc[b].loc["CZ0426"][wide_b.columns < 2016].mean()})
TEP = pd.DataFrame(rows)
TEP.to_csv(OUT / "teplice_pasma.csv", index=False)
mlad = TEP[TEP["pásmo"].isin(nac.BANDS[:6])]; star = TEP[TEP["pásmo"].isin(nac.BANDS[6:])]
print(f"průměrný efekt 45–74: {mlad['efekt Teplice p. b.'].mean():+.2f} p. b. ({mlad['efekt relativně k úrovni pásma %'].mean():+.0f} % úrovně), "
      f"75+: {star['efekt Teplice p. b.'].mean():+.2f} p. b. ({star['efekt relativně k úrovni pásma %'].mean():+.0f} % úrovně)")
TEP.round(3)

průměrný efekt 45–74: +2.51 p. b. (+11 % úrovně), 75+: +1.21 p. b. (+33 % úrovně)


,pásmo,efekt Teplice p. b.,placebo p (jednostranně),efekt relativně k úrovni pásma %
0,45–49,1.812,0.197,7.372
1,50–54,2.172,0.145,8.579
2,55–59,3.188,0.092,13.557
3,60–64,3.159,0.105,13.329
4,65–69,3.063,0.118,13.561
5,70–74,1.644,0.171,9.009
6,75–79,1.241,0.224,10.964
7,80–84,1.539,0.158,32.098
8,85+,0.856,0.145,56.795


## 5 · Shrnutí a co z toho plyne

In [7]:
r_v = lambda c, b: SKEN[(SKEN.kod == c) & (SKEN.podskupina == b)]["r ověření"].iloc[0]
intr = INT.set_index("kod")
SHR = pd.DataFrame([
    ("Vzdálenost k pracovišti", f"celek r = {cz(celek['vzdalenost_km'][0], 2)} (neprůkazné)",
     f"80–84: r = {cz(r_v('vzdalenost_km', '80–84'), 2)}, 85+: r = {cz(r_v('vzdalenost_km', '85+'), 2)}; interakce s 75+ q = {cz(intr.loc['vzdalenost_km', 'q interakce'], 4)}"),
    ("Vlastní pracoviště v okrese", f"celek r = {cz(celek['ma_mamocentrum'][0], 2)}",
     f"85+: r = {cz(r_v('ma_mamocentrum', '85+'), 2)}; interakce q = {cz(intr.loc['ma_mamocentrum', 'q interakce'], 4)}"),
    ("Podíl obyvatel velkých měst", f"celek r = {cz(celek['obce_4_pct'][0], 2)} (neprůkazné)",
     f"80+: r ≈ {cz(r_v('obce_4_pct', '85+'), 2)}; interakce q = {cz(intr.loc['obce_4_pct', 'q interakce'], 4)}"),
    ("Ženy s cizím občanstvím", f"celek r = {cz(celek['cizinky_pct'][0], 2)}",
     f"45–59 záporné (50–54: r = {cz(r_v('cizinky_pct', '50–54'), 2)}), 85+ kladné; interakce q = {cz(intr.loc['cizinky_pct', 'q interakce'], 4)}"),
    ("Prevence u praktika, změny v čase", "celek ≈ 0", f"žádné pásmo po korekci (min. q = {cz(DYN[DYN.expozice.str.startswith('prevence')].q.min(), 2)})"),
    ("Objednací lhůta, změny v čase", "celek ≈ 0", f"žádné pásmo po korekci (min. q = {cz(DYN[DYN.expozice.str.startswith('objednací')].q.min(), 2)})"),
    ("Otevření pracoviště (Teplice)", "celkem +2,2 p. b.",
     f"45–74: {cz(mlad['efekt Teplice p. b.'].mean(), 1)} p. b., 75+: {cz(star['efekt Teplice p. b.'].mean(), 1)} p. b."),
], columns=["ukazatel", "celá populace", "podskupina"])
SHR.to_csv(OUT / "shrnuti.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(SHR.style.hide(axis="index").set_properties(**{"text-align": "left"}))

ukazatel,celá populace,podskupina
Vzdálenost k pracovišti,"celek r = -0,18 (neprůkazné)","80–84: r = -0,47, 85+: r = -0,49; interakce s 75+ q = 0,0000"
Vlastní pracoviště v okrese,"celek r = 0,31","85+: r = 0,44; interakce q = 0,0098"
Podíl obyvatel velkých měst,"celek r = -0,06 (neprůkazné)","80+: r ≈ 0,51; interakce q = 0,0000"
Ženy s cizím občanstvím,"celek r = -0,24","45–59 záporné (50–54: r = -0,42), 85+ kladné; interakce q = 0,0000"
"Prevence u praktika, změny v čase",celek ≈ 0,"žádné pásmo po korekci (min. q = 0,79)"
"Objednací lhůta, změny v čase",celek ≈ 0,"žádné pásmo po korekci (min. q = 0,95)"
Otevření pracoviště (Teplice),"celkem +2,2 p. b.","45–74: 2,5 p. b., 75+: 1,2 p. b."


**Jak to číst.**

* **Dostupnost je problém nejstarších žen.** Vzdálenost k pracovišti, vlastní pracoviště v okrese,
  dojížďka i podíl obyvatel velkých měst s pokrytím u žen 45–74 skoro nesouvisejí, u žen 75+ silně
  (r kolem ±0,4 až 0,5). Rozdíl mezi pásmy je průkazný testem interakce i po korekci a drží v obou
  obdobích. Celkový průměr ho skrývá, protože ženy 75+ mají ve standardizovaném pokrytí 45+ jen
  asi pětinovou váhu.
* **Dvacet průkazných interakcí s věkem 75+ (kap. 2) není dvacet nálezů.** Většina ukazatelů
  (hustota, podíl velkých měst a malých obcí, ceny bytů, volební podíly, vzdělání) měří totéž:
  město vs. venkov. Starší ženy ve městech chodí víc, starší ženy na venkově méně; u mladších
  tento rozdíl není. Je to jeden vzorec, nejspíš dostupnost.
* **Ne všechno, co projde korekcí i replikací, je věrohodné.** Podíl hlasů pro STAN na Moravě
  (27 okresů) nemá mechanismus a nesouvisí s ničím dalším; bereme ho jako náhodu.
* **Dva ukazatele mění znaménko s věkem.** Podíl cizinek je spojen s nižším pokrytím u žen 45–59 a
  s vyšším u žen 85+; podíl žen mimo pracovní sílu naopak. V celku se tyto vztahy částečně ruší.
  U cizinek může jít o to, že starší cizinky v ČR jsou jiná skupina (dlouhodobě usazené) než
  mladší migrantky; to data neukážou.
* **Změny v čase podskupiny nepotvrzují.** Prevence u praktika ani objednací lhůta nepůsobí po
  korekci v žádném pásmu. Jediný nominálně průkazný odhad (75–79) je přesně ten typ falešného
  nálezu, kvůli kterému korekci děláme.
* **Predikce u Teplic se potvrdila jen napůl.** V absolutních bodech pomohlo otevření pracoviště
  víc ženám 45–74 (asi +2,5 p. b.) než ženám 75+ (+1,2 p. b.); vzhledem k jejich mnohem nižší
  výchozí úrovni ale ženám 75+ víc (+33 % proti +11 %). Žádné pásmo samo neprojde placebo testem
  (p 0,09–0,22) a jde o jediný případ. Predikci tedy ani nepotvrzuje, ani nevyvrací.

**Pro zadání VZP:**

* Segment **„dostupnostně omezené“** (brief, segment 7) má mít jako hlavní znak věk 75+ spolu se
  vzdáleností nebo bez pracoviště v okrese; u mladších žen vzdálenost sama segment netvoří.
* Intervence typu doprava, mobilní mamograf a termín u bližšího pracoviště (`docs/interventions.md`,
  položka 4) cílit na ženy 75+ ve vzdálených okresech; u žen 45–74 tam čekat malý účinek.
* Pro experiment na datech VZP **předregistrovat** interakci „intervence × věk 75+ × vzdálenost“
  jako primární podskupinovou hypotézu; ostatní podskupiny jen jako explorativní.
* Všechno je ekologické: okres se staršími ženami daleko od pracoviště není totéž co starší žena
  daleko od pracoviště. Na individuálních datech VZP to jde ověřit přímo (vzdálenost z obce
  bydliště, věk ženy).